In [5]:
!pip install -q google-genai pillow python-dotenv

In [6]:
model="gemini-2.5-flash",


In [7]:
model="gemini-2.5-flash",

In [8]:
!pip install -q google-genai pillow
import json, smtplib, getpass, os
from google import genai
from google.genai import types
from google.colab import files
from PIL import Image

print("=== Step 1: لاگ ان کی تفصیلات ===")
SENDER_EMAIL = input("اپنی جی میل آئی ڈی درج کریں: ").strip()
SENDER_PASSWORD = getpass.getpass("اپنا 16 ہندسوں کا App Password درج کریں: ").strip()
GEMINI_API_KEY = getpass.getpass("اپنی Gemini API Key درج کریں: ").strip()
CV_PATH = "Resume.pdf"  # یقینی بنائیں کہ CV کولاب میں اپ لوڈ ہے

client = genai.Client(api_key=GEMINI_API_KEY)
print("\n✅ سٹیپ 1: لاگ ان اور سیٹ اپ مکمل ہو گیا!")

=== Step 1: لاگ ان کی تفصیلات ===
اپنی جی میل آئی ڈی درج کریں: umairkhanbarki001@gmail.com 
اپنا 16 ہندسوں کا App Password درج کریں: ··········
اپنی Gemini API Key درج کریں: ··········

✅ سٹیپ 1: لاگ ان اور سیٹ اپ مکمل ہو گیا!


In [12]:
# 1. ضروری لائبریریز انسٹال کریں
!pip install -q google-genai pillow gradio

import json
import smtplib
import os
from email.mime.multipart import MIMEMultipart
from email.mime.text import MIMEText
from email.mime.application import MIMEApplication
from google import genai
from google.genai import types
from PIL import Image
import gradio as gr

# 2. ای میل بھیجنے اور جاب پوسٹ پراسیس کرنے کا مین فنکشن
def process_job_application(sender_email, sender_password, gemini_api_key, cv_file, screenshot_file):
    if not sender_email or not sender_password or not gemini_api_key:
        return "❌ **ایرر:** براہ کرم تمام لاگ ان تفصیلات (Gmail, App Password, Gemini API Key) درج کریں۔"
    if cv_file is None:
        return "❌ **ایرر:** براہ کرم اپنی CV (PDF) فائل اپ لوڈ کریں۔"
    if screenshot_file is None:
        return "❌ **ایرر:** براہ کرم جاب پوسٹ کا اسکرین شاٹ اپ لوڈ کریں۔"

    try:
        # Gemini Client کا آغاز
        client = genai.Client(api_key=gemini_api_key)
        img = Image.open(screenshot_file)

        # AI Vision Prompt - آپ کی تفصیلات کے ساتھ
        prompt = """
        Analyze this LinkedIn job post screenshot carefully and extract the following details.
        Return ONLY a valid JSON object in this exact format:
        {
            "hr_email": "extracted email address or null if not found",
            "job_title": "extracted position name",
            "company_name": "company name if mentioned, else 'Hiring Team'",
            "email_subject": "Application for [Job Title] - umair",
            "email_body": "Dear Hiring Manager,\\n\\nI am writing to express my strong interest in the position. With a background in Electrical Engineering (UET Peshawar) and professional experience in telecom site design and data management, I am confident in my ability to add value to your team.\\n\\nI have attached my resume for your review and would welcome the opportunity to discuss how my technical skills align with your requirements.\\n\\nSincerely,\\numair\\nMBB Design Engineer\\nPhone: 123456789011\\nLinkedIn: www.xyz"
        }
        """

        response = client.models.generate_content(
            model="gemini-3.8-flash",
            contents=[img, prompt],
            config=types.GenerateContentConfig(
                response_mime_type="application/json"
            ),
        )

        info = json.loads(response.text)
        hr_email = info.get("hr_email")

        if not hr_email or hr_email == "null":
            return "⚠️ **تنبیہ:** اسکرین شاٹ میں کوئی ای میل آئی ڈی نہیں مل سکی! براہ کرم کوئی دوسری تصویر اپ لوڈ کریں۔"

        # SMTP کے ذریعے ای میل بھیجنا
        msg = MIMEMultipart()
        msg['From'] = sender_email
        msg['To'] = hr_email
        msg['Subject'] = info.get("email_subject")

        msg.attach(MIMEText(info.get("email_body"), 'plain'))

        # CV (PDF) فائل منسلک کرنا
        with open(cv_file.name, "rb") as f:
            attach = MIMEApplication(f.read(), _subtype="pdf")
            attach.add_header('Content-Disposition', 'attachment', filename="Umair_CV.pdf")
            msg.attach(attach)

        server = smtplib.SMTP('smtp.gmail.com', 587)
        server.starttls()
        server.login(sender_email, sender_password)
        server.sendmail(sender_email, hr_email, msg.as_string())
        server.quit()

        return f"### ✅ **ای میل کامیابی سے بھیج دی گئی ہے!**\n\n**📧 HR Email:** {hr_email}\n\n**📌 Subject:** {info.get('email_subject')}\n\n**📝 Sent Cover Letter:**\n```\n{info.get('email_body')}\n```"

    except Exception as e:
        return f"❌ **ایرر (Error):** {str(e)}"

# 3. خوبصورت اور بولڈ UI تھیم بنانے کے لیے Custom CSS
custom_css = """
body { background-color: #0f172a; color: #f8fafc; }
.bold-header { font-weight: 800; color: #38bdf8; font-size: 1.8rem; margin-bottom: 0.5rem; }
.sub-header { font-size: 1rem; color: #94a3b8; font-weight: 600; }
.gr-button-primary { background: linear-gradient(135deg, #0284c7 0%, #2563eb 100%) !important; font-weight: bold !important; font-size: 1.1rem !important; border: none !important; }
.gr-button-primary:hover { background: linear-gradient(135deg, #0369a1 0%, #1d4ed8 100%) !important; }
label span { font-weight: bold !important; color: #e2e8f0 !important; font-size: 0.95rem !important; }
"""

# 4. Gradio Dashboard UI ڈیزائن
with gr.Blocks(css=custom_css, title="Auto Job Application System") as app:
    gr.Markdown("<div class='bold-header'>🚀 LINKEDIN AUTO JOB APPLICATION DASHBOARD</div>")
    gr.Markdown("<div class='sub-header'>اپنی CV اور اسکرین شاٹ اپ لوڈ کریں، AI آپ کے لیے HR کی ای میل نکال کر ڈائریکٹ ای میل بھیج دے گا۔</div><br>")

    with gr.Row():
        with gr.Column(scale=1):
            gr.Markdown("### 🔑 **1. کریڈنشیلز (Credentials)**")
            email_input = gr.Textbox(label="Sender Gmail ID", placeholder="your_email@gmail.com")
            password_input = gr.Textbox(label="Gmail App Password (16-digits)", type="password", placeholder="xxxx xxxx xxxx xxxx")
            api_key_input = gr.Textbox(label="Gemini API Key", type="password", placeholder="AIzaSy...")

            gr.Markdown("<br>### 📄 **2. فائلز اپ لوڈ کریں (Files)**")
            cv_input = gr.File(label="Upload Resume / CV (PDF)", file_types=[".pdf"])
            screenshot_input = gr.Image(label="Upload Job Screenshot (JPEG/PNG)", type="filepath")

            submit_btn = gr.Button("🚀 AI 분석 & EMAIL SEND NOW", variant="primary")

        with gr.Column(scale=1):
            gr.Markdown("### 📡 **3. سٹیٹس اور لائیو آؤٹ پٹ (Live Terminal)**")
            output_text = gr.Markdown(value="*سسٹم تیار ہے۔ فارم پرُ کر کے بٹن پر کلک کریں۔*", elem_id="output-box")

    submit_btn.click(
        fn=process_job_application,
        inputs=[email_input, password_input, api_key_input, cv_input, screenshot_input],
        outputs=output_text
    )

# لائیو ویب ایپ چلائیں (یہ اپ کو ایک آن لائن پبلک لنک بھی دے گی)
app.launch(share=True, debug=True)


/tmp/ipykernel_2537/784226318.py:92: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: css. Please pass these parameters to launch() instead.
  with gr.Blocks(css=custom_css, title="Auto Job Application System") as app:


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://3ab2576ac6ffa3c309.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://3ab2576ac6ffa3c309.gradio.live


In [13]:
# 1. Install Required Libraries
!pip install -q google-genai pillow gradio

import json
import smtplib
import os
from email.mime.multipart import MIMEMultipart
from email.mime.text import MIMEText
from email.mime.application import MIMEApplication
from google import genai
from google.genai import types
from PIL import Image
import gradio as gr

# 2. Main Function to Process Job Application and Send Email
def process_job_application(sender_email, sender_password, gemini_api_key, cv_file, screenshot_file):
    if not sender_email or not sender_password or not gemini_api_key:
        return "❌ **Error:** Please provide all credentials (Gmail, App Password, and Gemini API Key)."
    if cv_file is None:
        return "❌ **Error:** Please upload your resume / CV (PDF) file."
    if screenshot_file is None:
        return "❌ **Error:** Please upload the job post screenshot."

    try:
        # Initialize Gemini Client
        client = genai.Client(api_key=gemini_api_key)
        img = Image.open(screenshot_file)

        # AI Vision Prompt - Custom extracted details
        prompt = """
        Analyze this LinkedIn job post screenshot carefully and extract the following details.
        Return ONLY a valid JSON object in this exact format:
        {
            "hr_email": "extracted email address or null if not found",
            "job_title": "extracted position name",
            "company_name": "company name if mentioned, else 'Hiring Team'",
            "email_subject": "Application for [Job Title] - umair",
            "email_body": "Dear Hiring Manager,\\n\\nI am writing to express my strong interest in the position. With a background in Electrical Engineering (UET Peshawar) and professional experience in telecom site design and data management, I am confident in my ability to add value to your team.\\n\\nI have attached my resume for your review and would welcome the opportunity to discuss how my technical skills align with your requirements.\\n\\nSincerely,\\numair\\nMBB Design Engineer\\nPhone: 123456789011\\nLinkedIn: www.xyz"
        }
        """

        response = client.models.generate_content(
            model="gemini-3.8-flash",
            contents=[img, prompt],
            config=types.GenerateContentConfig(
                response_mime_type="application/json"
            ),
        )

        info = json.loads(response.text)
        hr_email = info.get("hr_email")

        if not hr_email or hr_email == "null":
            return "⚠️ **Warning:** No email address found in the screenshot! Please try uploading a different image."

        # Send Email via SMTP
        msg = MIMEMultipart()
        msg['From'] = sender_email
        msg['To'] = hr_email
        msg['Subject'] = info.get("email_subject")

        msg.attach(MIMEText(info.get("email_body"), 'plain'))

        # Attach CV (PDF) file
        with open(cv_file.name, "rb") as f:
            attach = MIMEApplication(f.read(), _subtype="pdf")
            attach.add_header('Content-Disposition', 'attachment', filename="Umair_CV.pdf")
            msg.attach(attach)

        server = smtplib.SMTP('smtp.gmail.com', 587)
        server.starttls()
        server.login(sender_email, sender_password)
        server.sendmail(sender_email, hr_email, msg.as_string())
        server.quit()

        return f"### ✅ **Email Sent Successfully!**\n\n**📧 HR Email:** {hr_email}\n\n**📌 Subject:** {info.get('email_subject')}\n\n**📝 Sent Cover Letter:**\n```\n{info.get('email_body')}\n```"

    except Exception as e:
        return f"❌ **Error:** {str(e)}"

# 3. Custom CSS for Sleek & Bold UI Theme
custom_css = """
body { background-color: #0f172a; color: #f8fafc; }
.bold-header { font-weight: 800; color: #38bdf8; font-size: 1.8rem; margin-bottom: 0.5rem; }
.sub-header { font-size: 1rem; color: #94a3b8; font-weight: 600; }
.gr-button-primary { background: linear-gradient(135deg, #0284c7 0%, #2563eb 100%) !important; font-weight: bold !important; font-size: 1.1rem !important; border: none !important; }
.gr-button-primary:hover { background: linear-gradient(135deg, #0369a1 0%, #1d4ed8 100%) !important; }
label span { font-weight: bold !important; color: #e2e8f0 !important; font-size: 0.95rem !important; }
"""

# 4. Gradio Dashboard UI Design
with gr.Blocks(css=custom_css, title="Auto Job Application System") as app:
    gr.Markdown("<div class='bold-header'>🚀 LINKEDIN AUTO JOB APPLICATION DASHBOARD</div>")
    gr.Markdown("<div class='sub-header'>Upload your CV and job screenshot, AI will extract the HR email and dispatch your application instantly.</div><br>")

    with gr.Row():
        with gr.Column(scale=1):
            gr.Markdown("### 🔑 **1. Credentials**")
            email_input = gr.Textbox(label="Sender Gmail ID", placeholder="your_email@gmail.com")
            password_input = gr.Textbox(label="Gmail App Password (16-digits)", type="password", placeholder="xxxx xxxx xxxx xxxx")
            api_key_input = gr.Textbox(label="Gemini API Key", type="password", placeholder="AIzaSy...")

            gr.Markdown("<br>### 📄 **2. Upload Files**")
            cv_input = gr.File(label="Upload Resume / CV (PDF)", file_types=[".pdf"])
            screenshot_input = gr.Image(label="Upload Job Screenshot (JPEG/PNG)", type="filepath")

            submit_btn = gr.Button("🚀 AI Analysis & Email Send Now", variant="primary")

        with gr.Column(scale=1):
            gr.Markdown("### 📡 **3. Status & Live Output (Live Terminal)**")
            output_text = gr.Markdown(value="*System is ready. Fill out the form and click the button.*", elem_id="output-box")

    submit_btn.click(
        fn=process_job_application,
        inputs=[email_input, password_input, api_key_input, cv_input, screenshot_input],
        outputs=output_text
    )

# Launch the live web application (generates a public link)
app.launch(share=True, debug=True)

/tmp/ipykernel_2537/243965324.py:92: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: css. Please pass these parameters to launch() instead.
  with gr.Blocks(css=custom_css, title="Auto Job Application System") as app:


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://669179c37149d28a2f.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://669179c37149d28a2f.gradio.live
